# Nettoyage de `inventory_adjustments.csv`

Ce notebook diagnostique et nettoie les ajustements de stock. Les corrections sont limitées aux erreurs de format certaines. Les incohérences entre tables sont signalées mais conservées lorsqu'elles peuvent faire partie du narratif des données.

## 0. Bibliothèques et chemins

In [ ]:
from pathlib import Path
import re
import pandas as pd

CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR / 'data' / 'logistics' if (CURRENT_DIR / 'data' / 'logistics').exists() else CURRENT_DIR
SOURCE_FILE = DATA_DIR / 'inventory_adjustments.csv'
OUTPUT_FILE = DATA_DIR / 'inventory_adjustments_cleaned.csv'

## 1. Chargement et aperçu général

In [ ]:
df_raw = pd.read_csv(SOURCE_FILE, dtype=str)
df = df_raw.copy()
print(f'Dimensions : {df.shape[0]} lignes et {df.shape[1]} colonnes')
display(df.head())
display(df.dtypes.rename('type initial'))

## 2. Valeurs manquantes

Les références vers une réception ou une expédition sont optionnelles. On distingue donc ces absences structurelles de la valeur manquante dans le rôle d'approbation.

In [ ]:
missing = pd.DataFrame({
    'nombre': df.isna().sum(),
    'pourcentage': (df.isna().mean() * 100).round(2)
})
display(missing.loc[missing['nombre'] > 0])
display(df.loc[df['approved_by_role'].isna()])

Le rôle manquant ne peut pas être déduit de manière certaine à partir du statut, du motif ou de l'établissement. Il reste donc vide. Les références absentes signifient simplement qu'aucun document lié n'a été renseigné.

## 3. Doublons

In [ ]:
print('Doublons exacts :', df.duplicated().sum())
print('Identifiants adjustment_number dupliqués :', df['adjustment_number'].duplicated().sum())

Aucun doublon n'est présent : aucune ligne n'est supprimée.

## 4. Contrôle et nettoyage des formats

### 4.1 Identifiants

In [ ]:
identifier_patterns = {
    'adjustment_number': r'^ADJ-\d{4}-\d{5}$',
    'facility_code': r'^FCL-\d{4}$',
    'related_receipt_number': r'^RCV-\d{4}-\d{5}$',
    'related_shipment_number': r'^SHP-\d{4}-\d{5}$',
}
for column, pattern in identifier_patterns.items():
    invalid = df[column].notna() & ~df[column].str.match(pattern, na=False)
    print(f'{column}: {invalid.sum()} format(s) invalide(s)')

### 4.2 Dates

Deux dates suivent des formats différents. Elles sont converties explicitement pour éviter une interprétation ambiguë.

In [ ]:
standard_date_pattern = r'^\d{4}-\d{2}-\d{2}$'
non_standard_dates = ~df['adjustment_date'].str.match(standard_date_pattern, na=False)
display(df.loc[non_standard_dates, ['adjustment_number', 'adjustment_date']])

def parse_adjustment_date(value):
    for date_format in ('%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y'):
        try:
            return pd.to_datetime(value, format=date_format)
        except ValueError:
            pass
    return pd.NaT

df['adjustment_date'] = df['adjustment_date'].map(parse_adjustment_date)
print('Dates non convertibles restantes :', df['adjustment_date'].isna().sum())

### 4.3 Quantité d'ajustement

Une valeur contient le suffixe `case`. Le suffixe correspond à la colonne `unit`, il peut donc être retiré sans perte d'information. Les quantités négatives sont valides : elles représentent des sorties ou corrections à la baisse.

In [ ]:
quantity_as_number = pd.to_numeric(df['adjustment_quantity'], errors='coerce')
bad_quantity_mask = quantity_as_number.isna() & df['adjustment_quantity'].notna()
display(df.loc[bad_quantity_mask, ['adjustment_number', 'adjustment_quantity', 'unit']])

embedded_unit = df['adjustment_quantity'].str.extract(r'[0-9]\s+([A-Za-z_]+)\s*$', expand=False)
unit_conflict = embedded_unit.notna() & embedded_unit.str.casefold().ne(df['unit'].str.casefold())
print('Unités intégrées en conflit avec la colonne unit :', unit_conflict.sum())

clean_quantity = (df['adjustment_quantity']
                  .str.strip()
                  .str.replace(',', '.', regex=False)
                  .str.replace(r'\s+[A-Za-z_]+\s*$', '', regex=True))
df['adjustment_quantity'] = pd.to_numeric(clean_quantity, errors='raise')
display(df['adjustment_quantity'].describe())

### 4.4 Variables catégorielles

In [ ]:
categorical_columns = ['item_category', 'unit', 'adjustment_reason', 'approved_by_role', 'adjustment_status']
for column in categorical_columns:
    print(f'\n{column}')
    display(df[column].value_counts(dropna=False))

for column in categorical_columns:
    df[column] = df[column].str.strip().str.casefold()

print('Occurrences finales de transfer posting correction :',
      df['adjustment_reason'].eq('transfer posting correction').sum())

## 5. Contrôle des références externes

In [ ]:
facilities = pd.read_csv(DATA_DIR / 'erp_facilities.csv', dtype=str)
receipts = pd.read_csv(DATA_DIR / 'goods_receipts.csv', dtype=str).drop_duplicates('receipt_number')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv', dtype=str).drop_duplicates('shipment_number')

unknown_facilities = ~df['facility_code'].isin(facilities['facility_code'])
unknown_receipts = df['related_receipt_number'].notna() & ~df['related_receipt_number'].isin(receipts['receipt_number'])
unknown_shipments = df['related_shipment_number'].notna() & ~df['related_shipment_number'].isin(shipments['shipment_number'])
print('Établissements inconnus :', unknown_facilities.sum())
print('Réceptions référencées introuvables :', unknown_receipts.sum())
print('Expéditions référencées introuvables :', unknown_shipments.sum())

## 6. Cohérences inter-tables à préserver

Une référence existante n'est pas nécessairement cohérente avec l'ajustement. Ces écarts peuvent révéler le narratif caché du projet : ils sont mesurés, mais surtout pas corrigés ou supprimés automatiquement.

In [ ]:
with_receipt = df.loc[df['related_receipt_number'].notna()].merge(
    receipts, left_on='related_receipt_number', right_on='receipt_number',
    how='left', suffixes=('_adjustment', '_receipt'), validate='many_to_one'
)
receipt_year = pd.to_numeric(with_receipt['receipt_number'].str.extract(r'RCV-(\d{4})')[0])
adjustment_year_receipt = with_receipt['adjustment_date'].dt.year
print('Références de réception :', len(with_receipt))
print('Établissement différent :', with_receipt['facility_code'].ne(with_receipt['receiving_facility_code']).sum())
print('Unité différente :', with_receipt['unit'].ne(with_receipt['received_unit'].str.casefold()).sum())
print('Réception d’une année future :', receipt_year.gt(adjustment_year_receipt).sum())

with_shipment = df.loc[df['related_shipment_number'].notna()].merge(
    shipments, left_on='related_shipment_number', right_on='shipment_number',
    how='left', suffixes=('_adjustment', '_shipment'), validate='many_to_one'
)
shipment_year = pd.to_numeric(with_shipment['shipment_number'].str.extract(r'SHP-(\d{4})')[0])
adjustment_year_shipment = with_shipment['adjustment_date'].dt.year
facility_unrelated = ~(
    with_shipment['facility_code'].eq(with_shipment['origin_facility_code']) |
    with_shipment['facility_code'].eq(with_shipment['destination_facility_code'])
)
print('\nRéférences d’expédition :', len(with_shipment))
print('Établissement absent du trajet :', facility_unrelated.sum())
print('Catégorie différente :', with_shipment['item_category_adjustment'].ne(with_shipment['item_category_shipment']).sum())
print('Unité différente :', with_shipment['unit'].ne(with_shipment['declared_unit'].str.casefold()).sum())
print('Expédition d’une année future :', shipment_year.gt(adjustment_year_shipment).sum())

Les références sont techniquement valides mais plusieurs sont temporellement ou matériellement surprenantes. Elles sont conservées comme signaux potentiels pour l'analyse ultérieure.

## 7. Contrôles finaux

In [ ]:
assert df.shape == (140, 11)
assert df['adjustment_number'].is_unique
assert not df.duplicated().any()
assert df['adjustment_date'].notna().all()
assert df['adjustment_quantity'].notna().all()
assert not df['adjustment_quantity'].eq(0).any()
assert df['approved_by_role'].isna().sum() == 1
assert not unknown_facilities.any()
assert not unknown_receipts.any()
assert not unknown_shipments.any()
assert set(df['adjustment_status']) <= {'posted', 'pending_review', 'approved'}
print('Tous les contrôles finaux sont passés.')
df.info()

## 8. Export du fichier nettoyé

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d', na_rep='')
print(f'Fichier créé : {OUTPUT_FILE}')
print(f'Dimensions finales : {df.shape[0]} lignes et {df.shape[1]} colonnes')